In [1]:
question_bank_df = spark.table(
    "demo.silver.question_bank"
)

question_bank_df.select(
    "question_id",
    "question_version",
    "domain",
    "topic",
    "subtopic",
    "created_at",
    "validation_status",
    "is_active"
).show(truncate=False)

+------------+----------------+----------------+-----------------+---------------------+-------------------+-----------------+---------+
|question_id |question_version|domain          |topic            |subtopic             |created_at         |validation_status|is_active|
+------------+----------------+----------------+-----------------+---------------------+-------------------+-----------------+---------+
|question_001|1               |Computer Science|Operating Systems|Virtual Memory       |2026-07-19 12:00:00|approved         |true     |
|question_002|1               |Computer Science|Operating Systems|Virtual Memory       |2026-07-19 12:02:00|approved         |true     |
|question_003|1               |Computer Science|Programming      |Recursion            |2026-07-20 10:00:00|approved         |true     |
|question_004|1               |Computer Science|Programming      |Recursion            |2026-07-20 10:03:00|approved         |true     |
|question_005|1               |Computer S

In [2]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    concat_ws,
    sha2,
    lit,
    min as spark_min,
    max as spark_max
)

def normalize_text(column_name):
    return lower(
        trim(
            regexp_replace(
                col(column_name),
                r"\s+",
                " "
            )
        )
    )

prepared_questions_df = (
    question_bank_df
    .filter(
        col("domain").isNotNull()
        & col("topic").isNotNull()
        & col("subtopic").isNotNull()
    )
    .withColumn(
        "normalized_domain",
        normalize_text("domain")
    )
    .withColumn(
        "normalized_topic",
        normalize_text("topic")
    )
    .withColumn(
        "normalized_subtopic",
        normalize_text("subtopic")
    )
)

In [3]:
domain_taxonomy_df = (
    prepared_questions_df
    .groupBy(
        "domain",
        "normalized_domain"
    )
    .agg(
        spark_min("created_at").alias("first_detected_at"),
        spark_max("created_at").alias("last_detected_at"),
        spark_max(
            col("is_active").cast("int")
        ).cast("boolean").alias("is_active")
    )
    .withColumn("topic", lit(None).cast("string"))
    .withColumn("subtopic", lit(None).cast("string"))
    .withColumn("concept_name", lit(None).cast("string"))
    .withColumn("normalized_topic", lit(None).cast("string"))
    .withColumn("normalized_subtopic", lit(None).cast("string"))
    .withColumn("normalized_concept_name", lit(None).cast("string"))
    .withColumn("taxonomy_level", lit("domain"))
    .withColumn("parent_taxonomy_id", lit(None).cast("string"))
    .withColumn("source_type", lit("question_bank"))
    .withColumn("validation_status", lit("approved"))
    .withColumn(
        "taxonomy_hash",
        sha2(
            concat_ws(
                "||",
                lit("domain"),
                col("normalized_domain")
            ),
            256
        )
    )
    .withColumn("taxonomy_id", col("taxonomy_hash"))
)

In [4]:
topic_taxonomy_df = (
    prepared_questions_df
    .groupBy(
        "domain",
        "topic",
        "normalized_domain",
        "normalized_topic"
    )
    .agg(
        spark_min("created_at").alias("first_detected_at"),
        spark_max("created_at").alias("last_detected_at"),
        spark_max(
            col("is_active").cast("int")
        ).cast("boolean").alias("is_active")
    )
    .withColumn("subtopic", lit(None).cast("string"))
    .withColumn("concept_name", lit(None).cast("string"))
    .withColumn("normalized_subtopic", lit(None).cast("string"))
    .withColumn("normalized_concept_name", lit(None).cast("string"))
    .withColumn("taxonomy_level", lit("topic"))
    .withColumn(
        "parent_taxonomy_id",
        sha2(
            concat_ws(
                "||",
                lit("domain"),
                col("normalized_domain")
            ),
            256
        )
    )
    .withColumn("source_type", lit("question_bank"))
    .withColumn("validation_status", lit("approved"))
    .withColumn(
        "taxonomy_hash",
        sha2(
            concat_ws(
                "||",
                lit("topic"),
                col("normalized_domain"),
                col("normalized_topic")
            ),
            256
        )
    )
    .withColumn("taxonomy_id", col("taxonomy_hash"))
)

In [5]:
subtopic_taxonomy_df = (
    prepared_questions_df
    .groupBy(
        "domain",
        "topic",
        "subtopic",
        "normalized_domain",
        "normalized_topic",
        "normalized_subtopic"
    )
    .agg(
        spark_min("created_at").alias("first_detected_at"),
        spark_max("created_at").alias("last_detected_at"),
        spark_max(
            col("is_active").cast("int")
        ).cast("boolean").alias("is_active")
    )
    .withColumn("concept_name", lit(None).cast("string"))
    .withColumn("normalized_concept_name", lit(None).cast("string"))
    .withColumn("taxonomy_level", lit("subtopic"))
    .withColumn(
        "parent_taxonomy_id",
        sha2(
            concat_ws(
                "||",
                lit("topic"),
                col("normalized_domain"),
                col("normalized_topic")
            ),
            256
        )
    )
    .withColumn("source_type", lit("question_bank"))
    .withColumn("validation_status", lit("approved"))
    .withColumn(
        "taxonomy_hash",
        sha2(
            concat_ws(
                "||",
                lit("subtopic"),
                col("normalized_domain"),
                col("normalized_topic"),
                col("normalized_subtopic")
            ),
            256
        )
    )
    .withColumn("taxonomy_id", col("taxonomy_hash"))
)

In [6]:
taxonomy_columns = [
    "taxonomy_id",
    "domain",
    "topic",
    "subtopic",
    "concept_name",
    "normalized_domain",
    "normalized_topic",
    "normalized_subtopic",
    "normalized_concept_name",
    "taxonomy_level",
    "parent_taxonomy_id",
    "source_type",
    "first_detected_at",
    "last_detected_at",
    "validation_status",
    "is_active",
    "taxonomy_hash"
]

question_taxonomy_df = (
    domain_taxonomy_df.select(taxonomy_columns)
    .unionByName(
        topic_taxonomy_df.select(taxonomy_columns)
    )
    .unionByName(
        subtopic_taxonomy_df.select(taxonomy_columns)
    )
)

In [7]:
question_taxonomy_df.select(
    "taxonomy_level",
    "domain",
    "topic",
    "subtopic",
    "parent_taxonomy_id",
    "validation_status"
).orderBy(
    "taxonomy_level",
    "domain",
    "topic",
    "subtopic"
).show(truncate=False)

print(
    "Question-bank taxonomy rows:",
    question_taxonomy_df.count()
)


+--------------+----------------+-----------------+---------------------+----------------------------------------------------------------+-----------------+
|taxonomy_level|domain          |topic            |subtopic             |parent_taxonomy_id                                              |validation_status|
+--------------+----------------+-----------------+---------------------+----------------------------------------------------------------+-----------------+
|domain        |Computer Science|NULL             |NULL                 |NULL                                                            |approved         |
|subtopic      |Computer Science|Operating Systems|Process Memory Layout|e3a962692f730e9c99e3c3950032800398d2bb5a3b08ed13020b599dd9b8a80a|approved         |
|subtopic      |Computer Science|Operating Systems|Virtual Memory       |e3a962692f730e9c99e3c3950032800398d2bb5a3b08ed13020b599dd9b8a80a|approved         |
|subtopic      |Computer Science|Programming      |Recursi

In [8]:
reference_materials_df = spark.table(
    "demo.silver.reference_materials"
)

reference_materials_df.select(
    "reference_id",
    "domain",
    "topic",
    "title",
    "import_time",
    "reliability_level",
    "is_active"
).show(truncate=False)

+-------------+----------------+-----------------+---------------------------------+-------------------+-----------------+---------+
|reference_id |domain          |topic            |title                            |import_time        |reliability_level|is_active|
+-------------+----------------+-----------------+---------------------------------+-------------------+-----------------+---------+
|reference_001|Computer Science|Operating Systems|Page Fault Definition            |2026-07-18 10:00:00|official         |true     |
|reference_002|Computer Science|Operating Systems|Page Fault Handling Steps        |2026-07-18 10:00:00|official         |true     |
|reference_003|Computer Science|Operating Systems|Purpose of Virtual Memory        |2026-07-18 11:30:00|official         |true     |
|reference_004|Computer Science|Programming      |Recursion and the Base Case      |2026-07-19 09:00:00|approved         |true     |
|reference_005|Computer Science|Programming      |Recursion Compared 

In [9]:
ai_insights_df = spark.table(
    "demo.silver.ai_extracted_insights"
)

validated_insights_df = spark.table(
    "demo.silver.validated_learning_insights"
)

concept_candidates_df = (
    ai_insights_df.alias("ai")
    .join(
        validated_insights_df.alias("v"),
        col("ai.insight_id") == col("v.insight_id"),
        "left"
    )
    .join(
        reference_materials_df.alias("r"),
        col("v.reference_id") == col("r.reference_id"),
        "left"
    )
    .select(
        col("ai.insight_id"),
        col("ai.dynamic_concept_name"),
        col("ai.extracted_at"),
        col("ai.extraction_confidence"),
        col("v.validation_status"),
        col("v.semantic_match_score"),
        col("v.reliability_score"),
        col("r.domain"),
        col("r.topic"),
        col("r.title").alias("matched_reference_title"),
        col("r.reliability_level")
    )
)

concept_candidates_df.orderBy(
    "domain",
    "topic",
    "dynamic_concept_name"
).show(truncate=False)

+----------------------------------------------------------------+--------------------+-------------------+---------------------+-----------------+--------------------+-----------------+----------------+-----------------+---------------------------------+-----------------+
|insight_id                                                      |dynamic_concept_name|extracted_at       |extraction_confidence|validation_status|semantic_match_score|reliability_score|domain          |topic            |matched_reference_title          |reliability_level|
+----------------------------------------------------------------+--------------------+-------------------+---------------------+-----------------+--------------------+-----------------+----------------+-----------------+---------------------------------+-----------------+
|7215f4912ad030943daea0c55e0341db76874b42ea87bbb5011537e664c8d3e2|Memory Layout       |2026-07-22 14:00:08|1.0                  |weak_match       |0.5                 |0.65      

In [10]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    regexp_replace,
    when,
    lit
)

def normalize_column(column):
    return lower(
        trim(
            regexp_replace(column, r"\s+", " ")
        )
    )

existing_topics_df = (
    question_taxonomy_df
    .filter(col("taxonomy_level") == "topic")
    .select(
        col("taxonomy_id").alias("existing_topic_taxonomy_id"),
        col("normalized_topic").alias("existing_topic_name")
    )
)

existing_subtopics_df = (
    question_taxonomy_df
    .filter(col("taxonomy_level") == "subtopic")
    .select(
        col("taxonomy_id").alias("existing_subtopic_taxonomy_id"),
        col("normalized_subtopic").alias("existing_subtopic_name")
    )
)

classified_concepts_df = (
    concept_candidates_df
    .withColumn(
        "normalized_dynamic_concept",
        normalize_column(col("dynamic_concept_name"))
    )

    .join(
        existing_topics_df,
        col("normalized_dynamic_concept")
        == col("existing_topic_name"),
        "left"
    )

    .join(
        existing_subtopics_df,
        col("normalized_dynamic_concept")
        == col("existing_subtopic_name"),
        "left"
    )

    .withColumn(
        "taxonomy_action",
        when(
            col("existing_topic_taxonomy_id").isNotNull(),
            lit("use_existing_topic")
        )
        .when(
            col("existing_subtopic_taxonomy_id").isNotNull(),
            lit("use_existing_subtopic")
        )
        .otherwise(
            lit("new_concept_candidate")
        )
    )

    .withColumn(
        "matched_existing_taxonomy_id",
        when(
            col("existing_topic_taxonomy_id").isNotNull(),
            col("existing_topic_taxonomy_id")
        ).otherwise(
            col("existing_subtopic_taxonomy_id")
        )
    )
)

classified_concepts_df.select(
    "dynamic_concept_name",
    "domain",
    "topic",
    "validation_status",
    "semantic_match_score",
    "taxonomy_action",
    "matched_existing_taxonomy_id"
).orderBy(
    "taxonomy_action",
    "dynamic_concept_name"
).show(truncate=False)

+--------------------+----------------+-----------------+-----------------+--------------------+---------------------+----------------------------------------------------------------+
|dynamic_concept_name|domain          |topic            |validation_status|semantic_match_score|taxonomy_action      |matched_existing_taxonomy_id                                    |
+--------------------+----------------+-----------------+-----------------+--------------------+---------------------+----------------------------------------------------------------+
|Base Case           |Computer Science|Programming      |validated        |0.7                 |new_concept_candidate|NULL                                                            |
|Memory Layout       |Computer Science|Operating Systems|weak_match       |0.5                 |new_concept_candidate|NULL                                                            |
|Page Fault          |Computer Science|Operating Systems|validated        |0.7  

In [11]:
from pyspark.sql.functions import (
    col,
    expr,
    when,
    lit,
    sha2,
    concat_ws,
    min as spark_min,
    max as spark_max
)

new_concept_candidates_df = (
    classified_concepts_df
    .filter(col("taxonomy_action") == "new_concept_candidate")
)

# כל ה-subtopics הקיימים, כדי לנסות למצוא הורה מדויק יותר
existing_subtopic_parents_df = (
    question_taxonomy_df
    .filter(col("taxonomy_level") == "subtopic")
    .select(
        col("taxonomy_id").alias("candidate_subtopic_parent_id"),
        col("normalized_subtopic").alias("candidate_subtopic_name"),
        col("normalized_domain").alias("candidate_domain"),
        col("normalized_topic").alias("candidate_topic")
    )
)

# כל ה-topics הקיימים, לשימוש כהורה חלופי
existing_topic_parents_df = (
    question_taxonomy_df
    .filter(col("taxonomy_level") == "topic")
    .select(
        col("taxonomy_id").alias("fallback_topic_parent_id"),
        col("normalized_domain").alias("topic_parent_domain"),
        col("normalized_topic").alias("topic_parent_name")
    )
)

concepts_with_parent_candidates_df = (
    new_concept_candidates_df.alias("c")

    # ניסיון למצוא subtopic שמופיע בשם חומר הייחוס
    .join(
        existing_subtopic_parents_df.alias("s"),
        (
            col("c.normalized_dynamic_concept").isNotNull()
            & (normalize_column(col("c.domain")) == col("s.candidate_domain"))
            & (normalize_column(col("c.topic")) == col("s.candidate_topic"))
            & expr(
                """
                instr(
                    lower(coalesce(matched_reference_title, '')),
                    candidate_subtopic_name
                ) > 0
                """
            )
        ),
        "left"
    )

    # מציאת topic כהורה חלופי
    .join(
        existing_topic_parents_df.alias("t"),
        (
            normalize_column(col("c.domain"))
            == col("t.topic_parent_domain")
        )
        & (
            normalize_column(col("c.topic"))
            == col("t.topic_parent_name")
        ),
        "left"
    )

    .withColumn(
        "selected_parent_taxonomy_id",
        when(
            col("candidate_subtopic_parent_id").isNotNull(),
            col("candidate_subtopic_parent_id")
        ).otherwise(
            col("fallback_topic_parent_id")
        )
    )

    .withColumn(
        "new_taxonomy_validation_status",
        when(
            (col("validation_status") == "validated")
            & (col("semantic_match_score") >= lit(0.7)),
            lit("approved")
        ).otherwise(
            lit("pending")
        )
    )
)

concepts_with_parent_candidates_df.select(
    "dynamic_concept_name",
    "topic",
    "matched_reference_title",
    "candidate_subtopic_name",
    "selected_parent_taxonomy_id",
    "validation_status",
    "semantic_match_score",
    "new_taxonomy_validation_status"
).orderBy(
    "dynamic_concept_name"
).show(truncate=False)

+--------------------+-----------------+---------------------------+-----------------------+----------------------------------------------------------------+-----------------+--------------------+------------------------------+
|dynamic_concept_name|topic            |matched_reference_title    |candidate_subtopic_name|selected_parent_taxonomy_id                                     |validation_status|semantic_match_score|new_taxonomy_validation_status|
+--------------------+-----------------+---------------------------+-----------------------+----------------------------------------------------------------+-----------------+--------------------+------------------------------+
|Base Case           |Programming      |Recursion and the Base Case|recursion              |da4bbec24d294be7e79f3037d748b8fb5c89cf0c7872a6a9b5f63b0169afb058|validated        |0.7                 |pending                       |
|Memory Layout       |Operating Systems|Process Memory Layout      |process memory layou

In [12]:
from pyspark.sql.functions import round as spark_round

concepts_with_parent_candidates_df = (
    concepts_with_parent_candidates_df

    .withColumn(
        "semantic_score_rounded",
        spark_round(
            col("semantic_match_score").cast("double"),
            2
        )
    )

    .withColumn(
        "new_taxonomy_validation_status",
        when(
            (col("validation_status") == "validated")
            & (col("semantic_score_rounded") >= lit(0.7)),
            lit("approved")
        ).otherwise(
            lit("pending")
        )
    )
)

In [13]:
concepts_with_parent_candidates_df.select(
    "dynamic_concept_name",
    "candidate_subtopic_name",
    "validation_status",
    "semantic_score_rounded",
    "new_taxonomy_validation_status"
).orderBy(
    "dynamic_concept_name"
).show(truncate=False)

+--------------------+-----------------------+-----------------+----------------------+------------------------------+
|dynamic_concept_name|candidate_subtopic_name|validation_status|semantic_score_rounded|new_taxonomy_validation_status|
+--------------------+-----------------------+-----------------+----------------------+------------------------------+
|Base Case           |recursion              |validated        |0.7                   |approved                      |
|Memory Layout       |process memory layout  |weak_match       |0.5                   |pending                       |
|Page Fault          |NULL                   |validated        |0.7                   |approved                      |
|Process Memory      |process memory layout  |validated        |0.7                   |approved                      |
+--------------------+-----------------------+-----------------+----------------------+------------------------------+



In [14]:
concept_taxonomy_df = (
    concepts_with_parent_candidates_df
    .select(
        col("domain"),
        col("topic"),
        col("dynamic_concept_name").alias("concept_name"),
        col("normalized_dynamic_concept").alias(
            "normalized_concept_name"
        ),
        col("extracted_at"),
        col("new_taxonomy_validation_status").alias(
            "validation_status"
        ),
        col("selected_parent_taxonomy_id").alias(
            "parent_taxonomy_id"
        )
    )
    .groupBy(
        "domain",
        "topic",
        "concept_name",
        "normalized_concept_name",
        "validation_status",
        "parent_taxonomy_id"
    )
    .agg(
        spark_min("extracted_at").alias("first_detected_at"),
        spark_max("extracted_at").alias("last_detected_at")
    )

    .withColumn(
        "normalized_domain",
        normalize_column(col("domain"))
    )
    .withColumn(
        "normalized_topic",
        normalize_column(col("topic"))
    )

    .withColumn(
        "subtopic",
        lit(None).cast("string")
    )
    .withColumn(
        "normalized_subtopic",
        lit(None).cast("string")
    )

    .withColumn(
        "taxonomy_level",
        lit("concept")
    )
    .withColumn(
        "source_type",
        lit("ai_extraction")
    )
    .withColumn(
        "is_active",
        lit(True)
    )

    .withColumn(
        "taxonomy_hash",
        sha2(
            concat_ws(
                "||",
                lit("concept"),
                col("normalized_domain"),
                col("normalized_topic"),
                col("normalized_concept_name")
            ),
            256
        )
    )
    .withColumn(
        "taxonomy_id",
        col("taxonomy_hash")
    )

    .select(taxonomy_columns)
)

In [15]:
complete_taxonomy_df = (
    question_taxonomy_df
    .unionByName(concept_taxonomy_df)
)

In [16]:
complete_taxonomy_df.select(
    "taxonomy_level",
    "domain",
    "topic",
    "subtopic",
    "concept_name",
    "validation_status",
    "parent_taxonomy_id"
).orderBy(
    when(col("taxonomy_level") == "domain", 1)
    .when(col("taxonomy_level") == "topic", 2)
    .when(col("taxonomy_level") == "subtopic", 3)
    .when(col("taxonomy_level") == "concept", 4)
    .otherwise(5),
    "domain",
    "topic",
    "subtopic",
    "concept_name"
).show(truncate=False)

print(
    "Complete taxonomy rows:",
    complete_taxonomy_df.count()
)

+--------------+----------------+-----------------+---------------------+--------------+-----------------+----------------------------------------------------------------+
|taxonomy_level|domain          |topic            |subtopic             |concept_name  |validation_status|parent_taxonomy_id                                              |
+--------------+----------------+-----------------+---------------------+--------------+-----------------+----------------------------------------------------------------+
|domain        |Computer Science|NULL             |NULL                 |NULL          |approved         |NULL                                                            |
|topic         |Computer Science|Operating Systems|NULL                 |NULL          |approved         |3af7fa6cc86c65a77274c1065a16dbe8d864aad92950628ce54693753bc84a7a|
|topic         |Computer Science|Programming      |NULL                 |NULL          |approved         |3af7fa6cc86c65a77274c1065a16dbe8d8

In [17]:
spark.sql("""
DELETE FROM demo.silver.content_taxonomy
""")

DataFrame[]

In [18]:
complete_taxonomy_df.writeTo(
    "demo.silver.content_taxonomy"
).append()

In [19]:
spark.sql("""
SELECT
    taxonomy_level,
    COUNT(*) AS row_count
FROM demo.silver.content_taxonomy
GROUP BY taxonomy_level
ORDER BY taxonomy_level
""").show()

+--------------+---------+
|taxonomy_level|row_count|
+--------------+---------+
|       concept|        4|
|        domain|        1|
|      subtopic|        3|
|         topic|        2|
+--------------+---------+



In [20]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT taxonomy_id) AS distinct_taxonomy_ids,
    COUNT(DISTINCT taxonomy_hash) AS distinct_taxonomy_hashes
FROM demo.silver.content_taxonomy
""").show()

+----------+---------------------+------------------------+
|total_rows|distinct_taxonomy_ids|distinct_taxonomy_hashes|
+----------+---------------------+------------------------+
|        10|                   10|                      10|
+----------+---------------------+------------------------+

